# Week 3 — Baseline Evaluation

Runs **FixedSpread**, **Avellaneda-Stoikov**, and **GLFT** (Proposition 5) inside ABIDES for `N_EPISODES` episodes each.

**Outputs**
- `experiments/w03_baselines/baseline_quote_skew.png` — Figure 1: bid-offset vs inventory curves for AS and GLFT
- `experiments/w03_baselines/baseline_pnl.png` — mean ± 1σ cumulative PnL across episodes
- `experiments/w03_baselines/results.json` — Sharpe, MAP, MDD (mean ± std across episodes)

Every future RL agent comparison plot will overlay on top of the Figure 1 curves.

In [1]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

sys.path.insert(0, str(Path("..").resolve()))

from baselines.fixed_spread import FixedSpreadBaseline
from baselines.avellaneda_stoikov import AvellanedaStoikovBaseline
from baselines.glft import GLFTBaseline
from envs.lob_env import LOBMarketMakingEnv, TICK_OFFSETS

OUT_DIR = Path("experiments/w03_baselines")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"Output directory: {OUT_DIR.resolve()}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Imports OK
Output directory: /home/lawre/distrl-market-maker/notebooks/experiments/w03_baselines


In [2]:
# Target: base spread ~10 ticks, max skew ~30 ticks at Q_max
# (leaving headroom below 80-tick ceiling)

TICK_OFFSETS = np.arange(0, 81)   # 0-80 ticks
TICK_SIZE    = 0.01
Q_MAX        = 10

# Step 1: fix gamma and target spread
GAMMA  = 1.0
target_base_ticks = 10
target_base_dollars = target_base_ticks * TICK_SIZE  # = $0.10

# Step 2: solve for kappa
# (2/gamma)*ln(1+gamma/kappa) = 0.10
# ln(1+1/kappa) = 0.05
# kappa = 1/(exp(0.05)-1) = 19.5
import numpy as np
KAPPA = 1 / (np.exp(target_base_dollars * GAMMA / 2) - 1)
print(f"KAPPA = {KAPPA:.1f}")   # ~19.5

# Step 3: fix sigma so skew at Q_max spans ~30 ticks
# Q_max * gamma * sigma^2 * 0.5 = 30 ticks * tick_size
# sigma^2 = 30 * 0.01 / (10 * 1 * 0.5) = 0.06
# sigma = 0.245
target_skew_ticks = 30
SIGMA_INIT = np.sqrt(
    target_skew_ticks * TICK_SIZE / (Q_MAX * GAMMA * 0.5)
)
print(f"SIGMA = {SIGMA_INIT:.4f}")

XI          = 0.0
adapt_sigma = False

KAPPA = 19.5
SIGMA = 0.2449


In [3]:
# ── Configuration ─────────────────────────────────────────────────────

N_EPISODES  = 20         # episodes per baseline — increase for smoother curves
EPISODE_LEN = 390        # steps per episode (1-minute bars, one trading day)
SEED        = 42         # base seed; episode i uses seed SEED + i
TICK_SIZE   = 0.01
Q_MAX       = 10
SIGMA_INIT  = 0.245
GAMMA       = 1.0
KAPPA       = 19.5       # calibrated so base spread ≈ 4-6 ticks
XI          = 0.0      # GLFT market-impact parameter

def make_baselines():
    """Construct fresh baseline instances (called once per episode)."""
    return {
        "FixedSpread": FixedSpreadBaseline(half_spread_ticks=2),
        "AS": AvellanedaStoikovBaseline(
            gamma=GAMMA, kappa=KAPPA, sigma=SIGMA_INIT,
            T=EPISODE_LEN, tick_size=TICK_SIZE, adapt_sigma=False,
        ),
        "GLFT": GLFTBaseline(
            gamma=GAMMA, kappa=KAPPA, sigma=SIGMA_INIT, xi=XI, A=1.0,
            T=EPISODE_LEN, Q_max=Q_MAX, tick_size=TICK_SIZE, adapt_sigma=False,
        ),
    }

# Verify base spread at kappa=40
import numpy as np
base = (2/GAMMA) * np.log(1 + GAMMA/KAPPA)
print(f"Base spread at kappa={KAPPA}: ${base:.4f} = {base/TICK_SIZE:.1f} ticks")
print(f"Running {N_EPISODES} episodes per baseline ({N_EPISODES * EPISODE_LEN} total steps each)")

Base spread at kappa=19.5: $0.1000 = 10.0 ticks
Running 20 episodes per baseline (7800 total steps each)


In [4]:
# ── Metric helpers ────────────────────────────────────────────────────

def compute_sharpe(step_pnls: np.ndarray) -> float:
    """Intraday Sharpe: mean(r) / std(r) * sqrt(T)."""
    mu, sig = np.mean(step_pnls), np.std(step_pnls)
    return float(mu / sig * np.sqrt(len(step_pnls))) if sig > 1e-10 else 0.0

def compute_map(inventories: np.ndarray) -> float:
    """Mean Absolute Position: (1/T) * Σ |q_t|."""
    return float(np.mean(np.abs(inventories)))

def compute_mdd(cum_pnls: np.ndarray) -> float:
    """Max Drawdown: min_t(cum_pnl_t - max_{s<=t} cum_pnl_s). Always ≤ 0."""
    return float(np.min(cum_pnls - np.maximum.accumulate(cum_pnls)))

def run_episode(baseline, seed: int):
    """
    Run one episode and return per-step arrays.

    Returns dict with:
        step_pnls   : spread_pnl + inventory mark-to-market per step
        cum_pnls    : cumulative PnL
        inventories : inventory at each step
        bid_offsets : bid tick offsets at each step
        ask_offsets : ask tick offsets at each step
    """
    env = LOBMarketMakingEnv(
        reward_type="asymmetric",
        episode_len=EPISODE_LEN,
        Q_max=Q_MAX,
        tick_size=TICK_SIZE,
        seed=seed,
    )
    baseline.reset()
    obs, info = env.reset(seed=seed)

    step_pnls, inventories, bid_offsets, ask_offsets = [], [], [], []
    cum_pnl  = 0.0
    cum_pnls = []
    prev_mid = info["mid_price"]
    prev_inv = 0

    terminated = truncated = False
    while not (terminated or truncated):
        action = baseline.act(obs, info)
        obs, reward, terminated, truncated, info = env.step(action)

        inv  = info["inventory"]
        mid  = info["mid_price"]

        # Step PnL = spread capture + inventory mark-to-market
        step_pnl = info["spread_pnl"] + prev_inv * (mid - prev_mid)
        cum_pnl += step_pnl

        step_pnls.append(step_pnl)
        cum_pnls.append(cum_pnl)
        inventories.append(inv)
        bid_offsets.append(int(TICK_OFFSETS[action[0]]))
        ask_offsets.append(int(TICK_OFFSETS[action[1]]))

        prev_mid = mid
        prev_inv = inv

    env.close()
    return {
        "step_pnls":   np.array(step_pnls),
        "cum_pnls":    np.array(cum_pnls),
        "inventories": np.array(inventories),
        "bid_offsets": np.array(bid_offsets),
        "ask_offsets": np.array(ask_offsets),
    }

print("Helpers defined")

Helpers defined


In [5]:
# ── Run N_EPISODES per baseline ───────────────────────────────────────
#
# all_episodes[name] = list of per-episode dicts (one per episode)
# Each episode uses a different seed so ABIDES generates different
# background agent sequences.

all_episodes = {}   # name → list of episode dicts

baseline_names = list(make_baselines().keys())

for name in baseline_names:
    print(f"\n{'='*50}")
    print(f"Running {name}  ({N_EPISODES} episodes)")
    print(f"{'='*50}")

    episodes = []
    for ep in range(N_EPISODES):
        # Construct fresh baseline each episode — resets sigma history etc.
        baseline = make_baselines()[name]
        ep_seed  = SEED + ep
        data     = run_episode(baseline, seed=ep_seed)
        episodes.append(data)

        sharpe = compute_sharpe(data["step_pnls"])
        map_   = compute_map(data["inventories"])
        print(
            f"  ep {ep+1:02d} | "
            f"final_pnl={data['cum_pnls'][-1]:+7.3f}  "
            f"sharpe={sharpe:+.3f}  "
            f"map={map_:.2f}  "
            f"inv_range=[{data['inventories'].min()},{data['inventories'].max()}]"
        )

    all_episodes[name] = episodes
    print(f"  Done — {N_EPISODES} episodes collected")

print("\nAll baselines complete.")


Running FixedSpread  (20 episodes)


  ep 01 | final_pnl=-37.620  sharpe=-3.000  map=5.71  inv_range=[-10,10]
  ep 02 | final_pnl=-35.630  sharpe=-3.743  map=4.35  inv_range=[-10,10]
  ep 03 | final_pnl=-47.520  sharpe=-3.597  map=6.24  inv_range=[-10,10]
  ep 04 | final_pnl=-31.770  sharpe=-3.294  map=5.27  inv_range=[-10,10]
  ep 05 | final_pnl=-24.885  sharpe=-2.355  map=4.88  inv_range=[-10,10]
  ep 06 | final_pnl=-42.290  sharpe=-3.138  map=4.56  inv_range=[-10,9]
  ep 07 | final_pnl=-37.500  sharpe=-2.834  map=3.62  inv_range=[-10,4]
  ep 08 | final_pnl=-42.220  sharpe=-3.541  map=6.42  inv_range=[-10,10]
  ep 09 | final_pnl=-47.395  sharpe=-3.412  map=5.04  inv_range=[-3,10]
  ep 10 | final_pnl=-28.200  sharpe=-2.027  map=3.29  inv_range=[-8,10]
  ep 11 | final_pnl=-41.705  sharpe=-3.564  map=5.50  inv_range=[-10,10]
  ep 12 | final_pnl=-23.815  sharpe=-2.509  map=4.88  inv_range=[-10,10]
  ep 13 | final_pnl=-30.330  sharpe=-3.087  map=4.63  inv_range=[-10,9]
  ep 14 | final_pnl=-24.815  sharpe=-2.599  map=4.93  in

In [6]:
# ── Aggregate metrics across episodes ─────────────────────────────────
#
# For each baseline, report mean ± std across episodes.
# Sharpe is computed per-episode then averaged (not on pooled steps).

results = {}

for name, episodes in all_episodes.items():
    sharpes    = [compute_sharpe(ep["step_pnls"])   for ep in episodes]
    maps_      = [compute_map(ep["inventories"])     for ep in episodes]
    mdds       = [compute_mdd(ep["cum_pnls"])        for ep in episodes]
    final_pnls = [float(ep["cum_pnls"][-1])          for ep in episodes]

    results[name] = {
        "sharpe_mean": float(np.mean(sharpes)),
        "sharpe_std":  float(np.std(sharpes)),
        "map_mean":    float(np.mean(maps_)),
        "map_std":     float(np.std(maps_)),
        "mdd_mean":    float(np.mean(mdds)),
        "mdd_std":     float(np.std(mdds)),
        "pnl_mean":    float(np.mean(final_pnls)),
        "pnl_std":     float(np.std(final_pnls)),
        "n_episodes":  len(episodes),
    }

results_path = OUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved → {results_path}")
print()
print(f"{'Baseline':<14} {'Sharpe':>14} {'MAP':>12} {'MDD':>16} {'PnL':>16}")
print("-" * 76)
for name, m in results.items():
    print(
        f"{name:<14} "
        f"{m['sharpe_mean']:>+7.3f}±{m['sharpe_std']:.3f}  "
        f"{m['map_mean']:>6.2f}±{m['map_std']:.2f}  "
        f"{m['mdd_mean']:>8.3f}±{m['mdd_std']:.3f}  "
        f"{m['pnl_mean']:>8.3f}±{m['pnl_std']:.3f}"
    )

Saved → experiments/w03_baselines/results.json

Baseline               Sharpe          MAP              MDD              PnL
----------------------------------------------------------------------------
FixedSpread     -3.163±0.693    5.10±0.85   -40.852±10.032   -37.470±10.287
AS              -4.536±1.251    1.88±0.39   -26.131±6.268   -25.113±6.895
GLFT            -0.858±1.324    1.28±1.18    -5.628±5.201    -1.115±3.335


In [7]:
# ── Build quote skew curves — pooled across all episodes ──────────────
#
# Pool (inventory, bid_offset) pairs from all episodes so every
# inventory level has many observations → smooth, reliable curves.

def quote_skew_curve(inventories, bid_offsets, Q_max, min_visits=5, boundary_buffer=1):
    """
    Pool all episodes and compute mean ± std bid_offset per inventory level.

    Returns (inv_levels, mean_offsets, std_offsets).
    boundary_buffer: exclude inventory levels within this many units of ±Q_max
    so the boundary fallback doesn't distort the curve.
    """
    groups = defaultdict(list)
    for inv, off in zip(inventories, bid_offsets):
        if abs(int(inv)) <= Q_max - boundary_buffer:   # exclude near-boundary
            groups[int(inv)].append(int(off))

    inv_levels   = sorted(k for k in groups if len(groups[k]) >= min_visits)
    mean_offsets = [np.mean(groups[k]) for k in inv_levels]
    std_offsets  = [np.std(groups[k])  for k in inv_levels]
    return np.array(inv_levels), np.array(mean_offsets), np.array(std_offsets)


skew_curves = {}
for name in ["AS", "GLFT"]:
    inv_lvls, mean_off, std_off = quote_skew_curve(
        inventories=[inv for ep in all_episodes[name] for inv in ep["inventories"]],
        bid_offsets=[off for ep in all_episodes[name] for off in ep["bid_offsets"]],
        Q_max=Q_MAX
    )
    skew_curves[name] = (inv_lvls, mean_off, std_off)
    total_obs = sum(len(ep["inventories"]) for ep in all_episodes[name])
    print(f"{name}: {total_obs} total observations, "
          f"inventory range [{inv_lvls.min()}, {inv_lvls.max()}], "
          f"{len(inv_lvls)} levels with ≥5 visits")
    if 0 in inv_lvls:
        print(f"  mean bid offset at q=0:  {mean_off[inv_lvls == 0][0]:.2f} ticks")
    if 5 in inv_lvls:
        print(f"  mean bid offset at q=+5: {mean_off[inv_lvls == 5][0]:.2f} ticks")
    if -5 in inv_lvls:
        print(f"  mean bid offset at q=-5: {mean_off[inv_lvls == -5][0]:.2f} ticks")

AS: 7800 total observations, inventory range [-8, 9], 18 levels with ≥5 visits
  mean bid offset at q=0:  6.80 ticks
  mean bid offset at q=+5: 15.08 ticks
  mean bid offset at q=-5: 9.16 ticks
GLFT: 7800 total observations, inventory range [-4, 9], 14 levels with ≥5 visits
  mean bid offset at q=0:  22.93 ticks
  mean bid offset at q=+5: 41.00 ticks


In [8]:
# ── Figure 1: bid-offset vs inventory (quote skew curves) ─────────────

COLORS = {
    "AS":          "#f59e0b",
    "GLFT":        "#06b6d4",
    "FixedSpread": "#6b7280",
    "bg":          "#0d1117",
    "panel":       "#161b22",
    "grid":        "#21262d",
    "text":        "#e6edf3",
    "subtext":     "#8b949e",
}

fixed_offset = 2   # FixedSpreadBaseline half_spread_ticks

fig, ax = plt.subplots(figsize=(9, 5.5), facecolor=COLORS["bg"])
ax.set_facecolor(COLORS["panel"])

# ── Fixed-spread reference line ───────────────────────────────────────
all_inv = np.concatenate([skew_curves[n][0] for n in ["AS", "GLFT"]])
x_min, x_max = int(all_inv.min()) - 1, int(all_inv.max()) + 1

ax.axhline(
    fixed_offset, color=COLORS["FixedSpread"], lw=1.2,
    ls="--", alpha=0.7, label="FixedSpread (floor)", zorder=2
)

# ── AS and GLFT skew curves ───────────────────────────────────────────
for name in ["AS", "GLFT"]:
    inv_lvls, mean_off, std_off = skew_curves[name]
    color = COLORS[name]

    ax.plot(
        inv_lvls, mean_off,
        color=color, lw=2.2, marker="o", ms=4,
        label=f"{name}  (n={N_EPISODES} eps)", zorder=4
    )
    ax.fill_between(
        inv_lvls,
        mean_off - std_off,
        mean_off + std_off,
        color=color, alpha=0.12, zorder=3
    )

# ── Reference lines ───────────────────────────────────────────────────
ax.axvline(0, color=COLORS["subtext"], lw=0.8, ls=":", alpha=0.6, zorder=1)

# ── Grid and spines ───────────────────────────────────────────────────
ax.set_xlim(x_min, x_max)
ax.set_ylim(bottom=0)
ax.grid(color=COLORS["grid"], lw=0.6, zorder=0)
for spine in ax.spines.values():
    spine.set_edgecolor(COLORS["grid"])
ax.tick_params(colors=COLORS["subtext"], labelsize=10)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── Labels ────────────────────────────────────────────────────────────
ax.set_xlabel("Inventory  q", color=COLORS["text"], fontsize=12, labelpad=8)
ax.set_ylabel("Bid offset  (ticks below mid)", color=COLORS["text"], fontsize=12, labelpad=8)
ax.set_title(
    f"Quote Skew: Bid Offset vs Inventory\n"
    f"AS and GLFT (Prop. 5)  ·  {N_EPISODES} ABIDES episodes each  ·  shaded = ±1σ",
    color=COLORS["text"], fontsize=13, pad=14, loc="left"
)

ax.legend(
    framealpha=0.15, facecolor=COLORS["bg"],
    edgecolor=COLORS["grid"], labelcolor=COLORS["text"],
    fontsize=11, loc="upper left"
)

fig.tight_layout(pad=1.5)
fig_path = OUT_DIR / "baseline_quote_skew.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight", facecolor=COLORS["bg"])
plt.close()
print(f"Saved Figure 1 → {fig_path}")

Saved Figure 1 → experiments/w03_baselines/baseline_quote_skew.png


In [9]:
# ── Supplementary: mean ± 1σ cumulative PnL across episodes ──────────
#
# Align all episodes to the same length (EPISODE_LEN) then plot
# mean with ±1σ shaded band.

fig2, ax2 = plt.subplots(figsize=(9, 4), facecolor=COLORS["bg"])
ax2.set_facecolor(COLORS["panel"])

line_styles = {"FixedSpread": "--", "AS": "-", "GLFT": "-"}

for name, episodes in all_episodes.items():
    color = COLORS[name]
    # Stack cum_pnls — all episodes have the same length (EPISODE_LEN)
    matrix = np.stack([ep["cum_pnls"] for ep in episodes])  # (N_EPISODES, T)
    mean   = matrix.mean(axis=0)
    std    = matrix.std(axis=0)

    steps = np.arange(len(mean))
    ax2.plot(
        steps, mean,
        color=color, ls=line_styles[name], lw=1.8,
        label=f"{name}  (μ={mean[-1]:+.2f}, n={N_EPISODES})", alpha=0.9
    )
    ax2.fill_between(
        steps, mean - std, mean + std,
        color=color, alpha=0.10
    )

ax2.axhline(0, color=COLORS["subtext"], lw=0.6, ls=":", alpha=0.5)
ax2.grid(color=COLORS["grid"], lw=0.5)
for spine in ax2.spines.values():
    spine.set_edgecolor(COLORS["grid"])
ax2.tick_params(colors=COLORS["subtext"], labelsize=10)
ax2.set_xlabel("Step", color=COLORS["text"], fontsize=11)
ax2.set_ylabel("Cumulative PnL  ($)", color=COLORS["text"], fontsize=11)
ax2.set_title(
    f"Cumulative PnL — mean ± 1σ across {N_EPISODES} ABIDES episodes per baseline",
    color=COLORS["text"], fontsize=12, pad=10, loc="left"
)
ax2.legend(
    framealpha=0.15, facecolor=COLORS["bg"],
    edgecolor=COLORS["grid"], labelcolor=COLORS["text"], fontsize=10
)
fig2.tight_layout(pad=1.5)

pnl_path = OUT_DIR / "baseline_pnl.png"
fig2.savefig(pnl_path, dpi=150, bbox_inches="tight", facecolor=COLORS["bg"])
plt.close()
print(f"Saved cumulative PnL plot → {pnl_path}")

Saved cumulative PnL plot → experiments/w03_baselines/baseline_pnl.png


In [10]:
# ── Final summary ─────────────────────────────────────────────────────

print("=" * 76)
print(f"Week 3 baseline evaluation complete  ({N_EPISODES} episodes per baseline)")
print("=" * 76)
print()
print(f"{'Baseline':<14} {'Sharpe (μ±σ)':>16} {'MAP (μ±σ)':>13} {'MDD (μ±σ)':>16} {'PnL (μ±σ)':>16}")
print("-" * 80)
for name, m in results.items():
    print(
        f"{name:<14} "
        f"{m['sharpe_mean']:>+7.3f}±{m['sharpe_std']:.3f}  "
        f"{m['map_mean']:>5.2f}±{m['map_std']:.2f}  "
        f"{m['mdd_mean']:>8.3f}±{m['mdd_std']:.3f}  "
        f"{m['pnl_mean']:>8.3f}±{m['pnl_std']:.3f}"
    )
print()
print("Files written:")
print(f"  {OUT_DIR}/results.json")
print(f"  {OUT_DIR}/baseline_quote_skew.png  ← Figure 1")
print(f"  {OUT_DIR}/baseline_pnl.png")

Week 3 baseline evaluation complete  (20 episodes per baseline)

Baseline           Sharpe (μ±σ)     MAP (μ±σ)        MDD (μ±σ)        PnL (μ±σ)
--------------------------------------------------------------------------------
FixedSpread     -3.163±0.693   5.10±0.85   -40.852±10.032   -37.470±10.287
AS              -4.536±1.251   1.88±0.39   -26.131±6.268   -25.113±6.895
GLFT            -0.858±1.324   1.28±1.18    -5.628±5.201    -1.115±3.335

Files written:
  experiments/w03_baselines/results.json
  experiments/w03_baselines/baseline_quote_skew.png  ← Figure 1
  experiments/w03_baselines/baseline_pnl.png
